# Fase 0 - Exploración del Dataset IroSvA
Análisis estadístico descriptivo de las tres variantes dialectales del corpus IroSvA (Ortega-Bueno et al., 2019): distribución de clases, longitud de texto y presencia de variables lingüísticas relevantes para detección de ironía.

## 1. Importación de librerías

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import emoji

## 2. Carga de las tres variantes dialectales

In [2]:
df_mx = pd.read_csv('../data/irosva.mx.training.csv')
df_es = pd.read_csv('../data/irosva.es.training.csv')
df_cu = pd.read_csv('../data/irosva.cu.training.csv')

print(f"Columnas: {df_mx.columns.tolist()}")
print(f"Tipos:    {df_mx.dtypes.to_dict()}")
print(df_mx.head(3))

Columnas: ['ID', 'TOPIC', 'IS_IRONIC', 'MESSAGE']
Tipos:    {'ID': dtype('O'), 'TOPIC': dtype('O'), 'IS_IRONIC': dtype('int64'), 'MESSAGE': dtype('O')}
                                 ID           TOPIC  IS_IRONIC  \
0  6424ee0864a0af40660686e135f5652b  asuntosConacyt          1   
1  f59978451dd7fb228830fed2ae00c3ef  asuntosConacyt          1   
2  280963c5eb0d162858caf3480a7ea08c  asuntosConacyt          1   

                                             MESSAGE  
0  Rica económicamente, pero muy pobre en objetiv...  
1  En algo tiene razón, mafias hay en todo, hasta...  
2  ¿De cuándo acá tan preocupados por la ciencia ...  


## 3. Validación de calidad: nulos y duplicados

In [3]:
for nombre, df in [('México', df_mx), ('España', df_es), ('Cuba', df_cu)]:
    nulos = df.isnull().sum().sum()
    dups  = df.duplicated(subset='MESSAGE').sum()
    print(f"{nombre}: {nulos} nulos | {dups} duplicados en MESSAGE")

México: 0 nulos | 1 duplicados en MESSAGE
España: 0 nulos | 2 duplicados en MESSAGE
Cuba: 0 nulos | 0 duplicados en MESSAGE


## 4. Distribución de clases por variante
El dataset presenta un desbalance 2:1 (no irónico:irónico) consistente en las tres variantes, documentado en Ortega-Bueno et al. (2019).

In [4]:
for nombre, df in [('México', df_mx), ('España', df_es), ('Cuba', df_cu)]:
    total      = len(df)
    ironico    = df['IS_IRONIC'].sum()
    no_ironico = total - ironico
    print(f"{nombre}: Total={total} | Irónico={ironico} ({ironico/total*100:.1f}%) | No irónico={no_ironico} ({no_ironico/total*100:.1f}%)")

México: Total=2400 | Irónico=800 (33.3%) | No irónico=1600 (66.7%)
España: Total=2400 | Irónico=800 (33.3%) | No irónico=1600 (66.7%)
Cuba: Total=2400 | Irónico=800 (33.3%) | No irónico=1600 (66.7%)


## 5. Longitud de texto por variante

In [5]:
for nombre, df in [('México', df_mx), ('España', df_es), ('Cuba', df_cu)]:
    df['longitud'] = df['MESSAGE'].apply(len)
    print(f"\n=== {nombre} ===")
    print(df['longitud'].describe().round(1))


=== México ===
count    2400.0
mean      114.6
std        73.8
min        14.0
25%        55.0
50%        96.0
75%       155.0
max       347.0
Name: longitud, dtype: float64

=== España ===
count    2400.0
mean      154.7
std        75.9
min        16.0
25%        95.0
50%       141.5
75%       210.0
max       400.0
Name: longitud, dtype: float64

=== Cuba ===
count    2400.0
mean      156.6
std        77.4
min         4.0
25%        91.0
50%       154.0
75%       219.0
max       337.0
Name: longitud, dtype: float64


## 6. Ejemplos de tweets irónicos y no irónicos

In [6]:
print("=== TWEETS IRÓNICOS ===")
for tweet in df_mx[df_mx['IS_IRONIC']==1]['MESSAGE'].head(5).values:
    print("-", tweet)

print("\n=== TWEETS NO IRÓNICOS ===")
for tweet in df_mx[df_mx['IS_IRONIC']==0]['MESSAGE'].head(5).values:
    print("-", tweet)

=== TWEETS IRÓNICOS ===
- Rica económicamente, pero muy pobre en objetividad.
- En algo tiene razón, mafias hay en todo, hasta en su 4t.
- ¿De cuándo acá tan preocupados por la ciencia y la investigación?
- De una vez que paren las titulaciones, que todos sean pasantes, así todos tendrán el mismo nivel que el director de Pemex y el subdirector de Conacyt 😡
- @LopesDorigaa @Dolores_PL  Es el que también te atiende tu tiendita.. a la Padierna...!!

=== TWEETS NO IRÓNICOS ===
- Lo más grave es que en la entrevista menciona que, el modelo científico que debemos seguir es el de Cuba. ¿De dónde salió ésta gente? Su visión del mundo es completamente anacrónica y desinformada.
- Y tanta gente capacitada que se fue a  la banca por causa de estos ineptos
- De nada; si se van a gastar mis impuestos que mejor que sea en algo que nos mejore como país.
- Ellos quieren juventudes atrapadas en la ignorancia, que no salgan y vean lo que pasa en el mundo, que no aprendan. El dominio de AMLO SOBRE LOS CI

## 7. Variables lingüísticas: mayúsculas enfáticas, puntuación exagerada y emojis
Ortega-Bueno et al. (2022) identifican mayúsculas enfáticas, puntuación exagerada y emojis como señales estilísticas relevantes para la detección de ironía en español. Se analiza su distribución por clase.

In [7]:
df_all = pd.concat([df_mx, df_es, df_cu], ignore_index=True)

# Mayúsculas enfáticas: palabras completamente en mayúsculas (mín. 3 caracteres)
df_all['mayuscula_enfatica'] = df_all['MESSAGE'].apply(
    lambda x: bool(re.search(r'\b[A-ZÁÉÍÓÚÜÑ]{3,}\b', x))
)
# Puntuación exagerada: dos o más signos consecutivos (!! ?? !?)
df_all['puntuacion_exagerada'] = df_all['MESSAGE'].apply(
    lambda x: bool(re.search(r'[!?]{2,}', x))
)
# Presencia de al menos un emoji
df_all['tiene_emoji'] = df_all['MESSAGE'].apply(
    lambda x: any(char in emoji.EMOJI_DATA for char in x)
)

for feature in ['mayuscula_enfatica', 'puntuacion_exagerada', 'tiene_emoji']:
    por_clase = df_all.groupby('IS_IRONIC')[feature].mean() * 100
    print(f"\n{feature}:")
    print(f"  No irónico: {por_clase[0]:.1f}%")
    print(f"  Irónico:    {por_clase[1]:.1f}%")


mayuscula_enfatica:
  No irónico: 17.6%
  Irónico:    19.1%

puntuacion_exagerada:
  No irónico: 7.1%
  Irónico:    15.0%

tiene_emoji:
  No irónico: 7.3%
  Irónico:    9.8%


## 8. Distribución de emojis por clase y variante

In [8]:
for nombre, df in [('México', df_mx), ('España', df_es), ('Cuba', df_cu)]:
    df['tiene_emoji'] = df['MESSAGE'].apply(lambda x: any(char in emoji.EMOJI_DATA for char in x))
    por_clase = df.groupby('IS_IRONIC')['tiene_emoji'].mean() * 100
    print(f"\n{nombre}:")
    print(f"  No irónico: {por_clase[0]:.1f}% con emoji")
    print(f"  Irónico:    {por_clase[1]:.1f}% con emoji")


México:
  No irónico: 7.9% con emoji
  Irónico:    16.6% con emoji

España:
  No irónico: 13.9% con emoji
  Irónico:    12.6% con emoji

Cuba:
  No irónico: 0.0% con emoji
  Irónico:    0.0% con emoji
